# 🛒 Unidad 1 – Ciencia de Datos
## Actividad: EDA y Limpieza de Datos de E-commerce
**TecNM Campus Apatzingán | Ingeniería en Sistemas Computacionales**

---

### Reflexión inicial
> Antes de ejecutar cualquier celda, responde aquí (en esta misma celda de texto):
>
> - ¿Qué tipo de problemas de calidad esperarías encontrar en un dataset de transacciones comerciales reales?
> - ¿Qué columnas crees que serán más problemáticas?
>
> *Escribe tu reflexión aquí...*

## Paso 1 – Carga de Librerías y Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 4)

# Carga directa desde UCI (puede tardar ~1 minuto por el tamaño del archivo)
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx'
print('Cargando dataset desde UCI...')
df = pd.read_excel(url)
print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
df.head()

## Paso 2 – Exploración Inicial

In [ ]:
# Dimensiones del dataset
print('Shape:', df.shape)

# Información general: tipos de datos y conteo de no-nulos
df.info()

In [ ]:
# Últimas 10 filas del dataset
df.tail(10)

### ✏️ Responde aquí las preguntas guía del Paso 2:

1. ¿Cuántas filas y columnas tiene el dataset?
2. ¿Qué columnas presentan valores faltantes según df.info()?
3. ¿Qué tipo de dato tiene InvoiceDate? ¿Es el correcto?

*Escribe tus respuestas aquí...*

## Paso 3 – Estadística Descriptiva Básica

In [ ]:
# Estadísticas descriptivas de todas las columnas numéricas
df.describe()

In [ ]:
# Media, mediana y moda de Quantity y UnitPrice
for col in ['Quantity', 'UnitPrice']:
    print(f'--- {col} ---')
    print(f'  Media:   {df[col].mean():.2f}')
    print(f'  Mediana: {df[col].median():.2f}')
    print(f'  Moda:    {df[col].mode()[0]:.2f}')
    print()

In [ ]:
# Histogramas de distribución
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['Quantity'].hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de Quantity')
axes[0].set_xlabel('Cantidad')

df['UnitPrice'].hist(bins=50, ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Distribución de UnitPrice')
axes[1].set_xlabel('Precio unitario (£)')

plt.tight_layout()
plt.show()

### ✏️ Responde aquí las preguntas guía del Paso 3:

1. ¿Cuál es la media y mediana de Quantity? ¿Son similares? ¿Qué indica eso?
2. ¿El valor mínimo de Quantity o UnitPrice tiene sentido? ¿Qué podría explicarlo?
3. ¿La distribución de UnitPrice es simétrica o asimétrica?

*Escribe tus respuestas aquí...*

## Paso 4 – Detección de Valores Nulos

In [ ]:
# Conteo y porcentaje de nulos por columna
nulos = df.isnull().sum()
pct_nulos = (nulos / len(df)) * 100
reporte_nulos = pd.DataFrame({'Nulos': nulos, 'Porcentaje (%)': pct_nulos.round(2)})
print(reporte_nulos[reporte_nulos['Nulos'] > 0])

In [ ]:
# Mapa de calor de valores nulos
plt.figure(figsize=(12, 4))
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.title('Mapa de valores nulos en el dataset')
plt.tight_layout()
plt.show()

### ✏️ Responde aquí las preguntas guía del Paso 4:

1. ¿Qué columnas tienen valores nulos y en qué porcentaje?
2. CustomerID tiene ~25% de nulos. ¿Es seguro eliminar esas filas? ¿Qué alternativa propondrías?
3. ¿Qué impacto tendría ignorar las filas con Description nula?

*Escribe tus respuestas aquí...*

## Paso 5 – Detección de Valores Erróneos

In [ ]:
# a) Detección de cancelaciones (InvoiceNo que inicia con 'C')
cancelaciones = df[df['InvoiceNo'].astype(str).str.startswith('C')]
print(f'Transacciones canceladas: {len(cancelaciones):,}')
print(f'Porcentaje del total: {len(cancelaciones)/len(df)*100:.2f}%')
cancelaciones.head(5)

In [ ]:
# b) Precios inválidos (UnitPrice <= 0)
precios_invalidos = df[df['UnitPrice'] <= 0]
print(f'Filas con UnitPrice <= 0: {len(precios_invalidos):,}')
precios_invalidos[['InvoiceNo', 'Description', 'Quantity', 'UnitPrice']].head(10)

In [ ]:
# c) Visualización de outliers en Quantity con boxplot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.boxplot(x=df['Quantity'], ax=axes[0], color='steelblue')
axes[0].set_title('Boxplot de Quantity (dataset completo)')

# Sin extremos para ver mejor la distribución central
q_filtrado = df[(df['Quantity'] > 0) & (df['Quantity'] < 200)]['Quantity']
sns.boxplot(x=q_filtrado, ax=axes[1], color='lightgreen')
axes[1].set_title('Boxplot de Quantity (0 < Q < 200)')

plt.tight_layout()
plt.show()

### ✏️ Responde aquí las preguntas guía del Paso 5:

1. ¿Cuántas transacciones corresponden a cancelaciones? ¿Las incluirías en un análisis de ventas?
2. ¿Qué harías con las filas de precio igual a 0? ¿Eliminarlas o conservarlas?
3. ¿Cuál es el valor máximo de Quantity? ¿Tiene sentido en ventas al mayoreo?

*Escribe tus respuestas aquí...*

## Paso 6 – Tratamiento y Limpieza de Datos

In [ ]:
df_clean = df.copy()

# 1. Eliminar filas con Description nula (0.27% del total)
df_clean = df_clean.dropna(subset=['Description'])

# 2. Separar cancelaciones
df_cancelaciones = df_clean[df_clean['InvoiceNo'].astype(str).str.startswith('C')].copy()
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]

# 3. Eliminar registros con UnitPrice <= 0
df_clean = df_clean[df_clean['UnitPrice'] > 0]

# 4. Crear variable de ingreso por línea
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

# Reporte antes / después
print(f'Registros originales:        {len(df):>10,}')
print(f'Registros limpios:           {len(df_clean):>10,}')
print(f'Registros eliminados:        {len(df)-len(df_clean):>10,}')
print(f'Cancelaciones separadas:     {len(df_cancelaciones):>10,}')

### ✏️ Responde aquí las preguntas guía del Paso 6:

1. ¿Qué porcentaje de registros se perdió tras la limpieza? ¿Es aceptable?
2. ¿El tratamiento de CustomerID nulo depende del objetivo del análisis? Explica.
3. ¿Qué otros problemas de calidad detectaste que no están cubiertos en el pipeline anterior?

*Escribe tus respuestas aquí...*

> 💡 **Continúa tú:** agrega aquí las decisiones de limpieza adicionales que propones, con su justificación.

## Paso 7 – Encoding de Variables Categóricas

In [ ]:
# Países únicos en el dataset limpio
print(f'Total de países distintos: {df_clean["Country"].nunique()}')
print(df_clean['Country'].value_counts())

In [ ]:
# a) Label Encoding
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_clean['Country_Label'] = le.fit_transform(df_clean['Country'])
print('Muestra de Label Encoding:')
print(df_clean[['Country', 'Country_Label']].drop_duplicates().sort_values('Country_Label').head(10))

In [ ]:
# b) One-Hot Encoding
country_dummies = pd.get_dummies(df_clean['Country'], prefix='Country')
print(f'Columnas generadas por One-Hot Encoding: {country_dummies.shape[1]}')
country_dummies.head(3)

### ✏️ Responde aquí las preguntas guía del Paso 7:

1. ¿Por qué el One-Hot Encoding genera tantas columnas nuevas?
2. ¿En qué situación preferirías Label Encoding sobre One-Hot? Da un ejemplo.
3. ¿Cuál usarías para Country en este dataset? Justifica tu respuesta.

*Escribe tus respuestas aquí...*

> 💡 **Continúa tú:** aplica encoding a otra columna del dataset que consideres relevante y documenta tu razonamiento.

## Paso 8 – Análisis de Correlación e Insights

In [ ]:
# Matriz de correlación de variables numéricas
numericas = df_clean[['Quantity', 'UnitPrice', 'TotalPrice']]
plt.figure(figsize=(6, 4))
sns.heatmap(numericas.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriz de correlación')
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 países por número de transacciones
top_paises = df_clean['Country'].value_counts().head(10)
top_paises.plot(kind='bar', color='steelblue', figsize=(10, 4), edgecolor='white')
plt.title('Top 10 países por número de transacciones')
plt.ylabel('Número de transacciones')
plt.xlabel('País')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### ✏️ Escribe aquí tus 3 Insights

Cada insight debe tener:
- **Hallazgo:** qué observaste (1-2 líneas)
- **Evidencia:** referencia a la gráfica o tabla que lo soporta
- **Recomendación:** qué acción tomaría la empresa

---
**Insight 1:**

*Escribe aquí...*

---
**Insight 2:**

*Escribe aquí...*

---
**Insight 3:**

*Escribe aquí...*

---

> 💡 **Continúa tú:** genera más visualizaciones y agrega insights adicionales. Propón tus propias preguntas de investigación sobre el dataset.